# 🎯 [PRC07_M04] NLP 및 지식 그래프 기반 대체 가능 호환 부품(Alternative Parts) 자동 추천 모델 고도화 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m04_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M04 |
| **미션 주제** | NLP 및 지식 그래프 기반 대체 가능 호환 부품(Alternative Parts) 자동 추천 모델 고도화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip() + "\n")
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 04. NLP 및 지식 그래프 기반 대체 가능 호환 부품(Alternative Parts) 자동 추천 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

지식 그래프 기반 대체 가능 호환 부품(Alternative Parts) 자동 추천 모델을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **텍스트 유사도와 지식 그래프, 규격 적합성을 결합해 해외 부품별 국산 대체 후보를 추천하고 추천 근거를 제시한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **부품 설명문의 TF-IDF 유사도, 전압·전력·온도·크기·인터페이스 6개 규격 적합성, 동의어를 간선으로 연결한 지식 그래프 이웃 유사도를 가중합해 부품별 Top-3 후보 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ai/prc07_intelligence_module.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추천 사유의 합리성 및 실무 수용성

#### 미션 PASS 조건

- 정량 기준: 추천/분류 정확도 85% 이상 달성


- 진행 기준: AI 추론 모듈 구현 및 벤치마크 테스트 완료


- 검증 방법: 과거 승인/처리 완료 데이터와의 추천 일치도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""부품 설명문의 TF-IDF 텍스트 유사도, 규격 적합성, 동의어를 연결한 지식 그래프 이웃 유사도를 결합해 해외 부품별 국산 대체 후보 Top-3를 추천하고, 과거 승인 이력 기준 정확도·요소별 기여·오추천 유형·추천 사유 문장을 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    # [TODO : 학생 실습 4] NLP 및 지식 그래프 기반 대체 가능 호환 부품(Alternative Parts) 자동 추천 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _sig(x):
    # [TODO : 학생 실습 4] NLP 및 지식 그래프 기반 대체 가능 호환 부품(Alternative Parts) 자동 추천 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# ---- 대체 부품 추천 공통 함수: 텍스트 유사도 + 규격 적합성 + 지식 그래프 이웃 유사도 ----
WEIGHTS = {"text": 0.35, "spec": 0.45, "graph": 0.20}


def spec_fit(p, c):
    """규격 적합성 0~1: 6개 조건 충족 비율. 전압·온도 한계(필수 조건)를 못 맞추면 점수를 절반으로 깎는다."""
    conds = [p["v"] == c["v"], c["w"] >= p["w"], c["tmin"] <= p["tmin"], c["tmax"] >= p["tmax"],
             abs(c["size_mm"] - p["size_mm"]) / p["size_mm"] <= 0.10, p["iface"] == c["iface"]]
    s = sum(conds) / len(conds)
    return (s * 0.5 if not (conds[0] and conds[2] and conds[3]) else s), conds


def graph_nodes(row):
    """지식 그래프에서 부품 노드와 연결된 이웃 노드 집합(범주·표준·인터페이스·기능어·등급)."""
    words = set(row["desc"].split()[1:-4])
    grade = "군용등급" if row["tmin"] <= -40 else "상용등급"
    return {f"cat:{row['category']}", f"std:{row['std']}", f"if:{row['iface']}", f"grade:{grade}", f"v:{row['v']}"} | {f"w:{w}" for w in words}


def build_graph(catalog):
    """동의어 사전을 그래프 간선(기능어 = 동의어)으로 넣어 서술이 달라도 같은 기능 노드로 연결되게 한다."""
    syn = {f"w:{b}": f"w:{a}" for c in CATS.values() for a, b in c["syn"].items()}
    canon = lambda nodes: {syn.get(x, x) for x in nodes}
    return {"canon": canon, "nodes": [canon(graph_nodes(r)) for _, r in catalog.iterrows()]}


def graph_sim(g, pn, cn):
    """두 부품이 공유하는 이웃 노드 비율(Jaccard)."""
    pn = g["canon"](pn)
    return len(pn & cn) / max(len(pn | cn), 1)


def score_table(parts_df, catalog):
    """해외 부품 x 같은 범주 국내 후보 쌍마다 텍스트·규격·그래프 점수를 계산한 표(가중치 적용 전)."""
    vec = TfidfVectorizer(token_pattern=r"[^ ]+").fit(list(parts_df["desc"]) + list(catalog["desc"]))
    text = cosine_similarity(vec.transform(parts_df["desc"]), vec.transform(catalog["desc"]))
    g = build_graph(catalog)
    rows = []
    for i, (_, p) in enumerate(parts_df.iterrows()):
        pn = graph_nodes(p)
        for j, (_, c) in enumerate(catalog.iterrows()):
            if c["category"] != p["category"]:
                continue
            sf, conds = spec_fit(p, c)
            rows.append({"part_no": p["part_no"], "cand_no": c["cand_no"], "vendor": c["vendor"], "text": text[i, j],
                         "spec": sf, "graph": graph_sim(g, pn, g["nodes"][j]),
                         "unmet": ",".join(n for n, ok in zip(["전압", "전력", "저온", "고온", "크기", "인터페이스"], conds) if not ok) or "-",
                         "kind": c["kind"], "correct": c["approved_for"] == p["part_no"]})
    return pd.DataFrame(rows)


def recommend(table, weights=None, top=3):
    """가중합 점수로 부품별 상위 top개 후보를 고른다."""
    w = weights or WEIGHTS
    t = table.assign(score=w["text"] * table["text"] + w["spec"] * table["spec"] + w["graph"] * table["graph"])
    return t.sort_values(["part_no", "score"], ascending=[True, False]).groupby("part_no").head(top)


def hit_rate(recs, k=1):
    """과거 승인 이력(정답 대체품)이 상위 k개 안에 들어간 부품 비율."""
    return recs.groupby("part_no").apply(lambda d: d["correct"].head(k).any(), include_groups=False).mean()

parts, catalog = make_parts(100, seed=7)
over = parts[parts["overseas"]].reset_index(drop=True)
print(f"[데이터] 해외 조달 부품 {len(over)}종, 국내 후보 카탈로그 {len(catalog)}건(승인 이력 있는 대체품 {int((catalog['kind'] == 'equivalent').sum())}건, 규격 불일치 유사 후보 {int((catalog['kind'] != 'equivalent').sum())}건)")

t0 = time.perf_counter()
table = score_table(over, catalog)
rec = recommend(table)
ms = (time.perf_counter() - t0) * 1000 / len(over)
print(f"추천 처리: 부품 1종당 {ms:.1f}ms (후보 쌍 {len(table)}개 점수 계산 포함)")

# 1) 요소별 기여: 어느 신호를 쓰느냐에 따라 정확도가 어떻게 달라지는가
combos = {"텍스트 유사도만": {"text": 1, "spec": 0, "graph": 0}, "규격 적합성만": {"text": 0, "spec": 1, "graph": 0},
          "그래프만": {"text": 0, "spec": 0, "graph": 1}, "텍스트+그래프": {"text": .5, "spec": 0, "graph": .5},
          "텍스트+규격": {"text": .4, "spec": .6, "graph": 0}, "세 신호 결합(운용)": WEIGHTS}
rows = []
for name, w in combos.items():
    r = recommend(table, w)
    bad = (r.groupby("part_no").head(1)["kind"] != "equivalent").mean()
    rows.append({"방식": name, "Top-1 정확도": hit_rate(r, 1), "Top-3 정확도": hit_rate(r, 3), "1순위가 규격불일치 후보인 비율": bad})
ab = pd.DataFrame(rows)
print("\n[요소별 기여(과거 승인 대체품을 상위에 올렸는가)]")
print(ab.round(3).to_string(index=False))
acc1, acc3 = hit_rate(rec, 1), hit_rate(rec, 3)
print(f"\n운용 방식 Top-1 정확도 {acc1:.1%}, Top-3 정확도 {acc3:.1%} (목표 85% 이상: {'달성' if acc1 >= 0.85 else '미달'})")

# 2) 오추천 분석: 1순위가 정답이 아닌 부품
top1 = rec.groupby("part_no").head(1)
wrong = top1[~top1["correct"]]
print(f"\n[1순위 오추천 {len(wrong)}건 분석]")
for _, w in wrong.iterrows():
    right = table[(table["part_no"] == w["part_no"]) & table["correct"]].iloc[0]
    if w["kind"] != "equivalent":
        why = f"규격 불일치 후보가 상위(미충족 {w['unmet']}) -> 규격 가중치 보강 필요"
    elif w["unmet"] == "-":
        why = "다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보"
    else:
        why = f"다른 부품용 승인 대체품이 상위(정답 후보는 크기 차이로 감점) -> 크기 허용 오차 재검토"
    print(f"  {w['part_no']}: 추천 {w['cand_no']}(점수 {w['score']:.2f}, {w['kind']}) / 정답 {right['cand_no']}(규격 미충족 {right['unmet']}) -> {why}")

# 3) 추천 사유 문장(실무 수용성: 근거를 사람이 읽을 수 있게)
print("\n[추천 사유 예시]")
for pn in over["part_no"].head(3):
    r = rec[rec["part_no"] == pn].iloc[0]
    p = parts[parts["part_no"] == pn].iloc[0]
    ok = "규격 6개 항목 모두 충족" if r["unmet"] == "-" else f"규격 미충족 항목 {r['unmet']}(기술 검토 필요)"
    print(f"  {pn} {p['category']} -> {r['cand_no']}({r['vendor']}): 설명 유사도 {r['text']:.2f}, 규격 적합성 {r['spec']:.2f}, 그래프 연결 {r['graph']:.2f} / {ok}")
print("※ 추천은 후보 제시이며 규격 적합성 최종 판단과 계약 의뢰는 기술검토관·계약 담당자가 결정한다.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(ab)); ax[0].bar(x - .2, ab["Top-1 정확도"], .4, label="top-1"); ax[0].bar(x + .2, ab["Top-3 정확도"], .4, label="top-3")
ax[0].set_xticks(x, ["text", "spec", "graph", "text+graph", "text+spec", "all 3"], rotation=20); ax[0].axhline(.85, color="r", ls="--"); ax[0].legend(); ax[0].set_title("Accuracy by signal combination")
allsc = recommend(table, WEIGHTS, 999)
ax[1].hist([allsc[allsc["correct"]]["score"], allsc[~allsc["correct"]]["score"]], bins=15, label=["approved", "other"], stacked=True)
ax[1].legend(); ax[1].set_title("Score distribution (candidates)"); ax[1].set_xlabel("weighted score")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[데이터] 해외 조달 부품 85종, 국내 후보 카탈로그 255건(승인 이력 있는 대체품 85건, 규격 불일치 유사 후보 170건)
추천 처리: 부품 1종당 14.0ms (후보 쌍 2745개 점수 계산 포함)

[요소별 기여(과거 승인 대체품을 상위에 올렸는가)]
         방식  Top-1 정확도  Top-3 정확도  1순위가 규격불일치 후보인 비율
   텍스트 유사도만      0.000      0.000              1.000
    규격 적합성만      0.812      1.000              0.000
       그래프만      0.553      0.741              0.047
    텍스트+그래프      0.000      0.082              1.000
     텍스트+규격      0.871      0.988              0.000
세 신호 결합(운용)      0.894      0.976              0.012

운용 방식 Top-1 정확도 89.4%, Top-3 정확도 97.6% (목표 85% 이상: 달성)

[1순위 오추천 9건 분석]
  P-0009: 추천 D-0187(점수 0.77, equivalent) / 정답 D-0016(규격 미충족 -) -> 다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보
  P-0028: 추천 D-0046(점수 0.76, equivalent) / 정답 D-0070(규격 미충족 크기) -> 다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보
  P-0032: 추천 D-0034(점수 0.82, equivalent) / 정답 D-0082(규격 미충족 크기) -> 다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보
  P-0046: 추천 D-0119(점수 0.67, volt_mismatch) / 정답 D-0118(규격 미충족 크기) -> 규격 불일치 후보가 상위(미충족 전압) -> 규격 가중치 보강 필요
  P-0049: 추천 D-0016(점수 0.82, equivalent) / 정답 D-0124(규격 미충족 -) -> 다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보
  P-0056: 추천 D-0058(점수 0.84, equivalent) / 정답 D-0145(규격 미충족 -) -> 다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보
  P-0060: 추천 D-0235(점수 0.81, equivalent) / 정답 D-0154(규격 미충족 -) -> 다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보
  P-0063: 추천 D-0181(점수 0.76, equivalent) / 정답 D-0163(규격 미충족 -) -> 다른 부품용으로 승인된 같은 범주 대체품이 규격도 충족 -> 승인 이력상 오답, 실무상 검토 후보
  P-0077: 추천 D-0049(점수 0.76, equivalent) / 정답 D-0196(규격 미충족 크기) -> 다른 부품용 승인 대체품이 상위(정답 후보는 크기 차이로 감점) -> 크기 허용 오차 재검토

[추천 사유 예시]
  P-0002 커넥터 -> D-0001(가상정밀(주)): 설명 유사도 0.64, 규격 적합성 1.00, 그래프 연결 1.00 / 규격 6개 항목 모두 충족
  P-0003 유압밸브 -> D-0004(가상정밀(주)): 설명 유사도 0.47, 규격 적합성 1.00, 그래프 연결 1.00 / 규격 6개 항목 모두 충족
  P-0004 베어링 -> D-0007(가상정밀(주)): 설명 유사도 0.54, 규격 적합성 1.00, 그래프 연결 1.00 / 규격 6개 항목 모두 충족
※ 추천은 후보 제시이며 규격 적합성 최종 판단과 계약 의뢰는 기술검토관·계약 담당자가 결정한다.
```
</details>
